# LeetCode 3438 — Find Valid Pair of Adjacent Digits in String bằng Python

Link đề: https://leetcode.com/problems/find-valid-pair-of-adjacent-digits-in-string/ · Độ khó: **Easy** · Chủ đề: Hash Table (bảng băm), String (chuỗi), Counting (đếm)

Skeleton LeetCode:

```python
class Solution:
    def findValidPair(self, s: str) -> str:
        ...
```

Ghi chú: ba ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (so sánh năm cách làm với nhau và với một bộ vét cạn viết riêng theo từng chữ của đề). Ví dụ nào do mình tự dựng sẽ ghi rõ "tự dựng" — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong chuỗi chữ số `s`, một **cặp hợp lệ** là hai chữ số **đứng cạnh nhau**, **khác nhau**, và mỗi chữ số trong cặp xuất hiện **trong toàn chuỗi `s` đúng bằng giá trị số của nó** (chữ số `'3'` phải xuất hiện đúng `3` lần). Trả về **cặp hợp lệ đầu tiên** khi duyệt từ trái sang phải, hoặc chuỗi rỗng `""` nếu không có.
2. Nắm các công cụ Python: duyệt chuỗi theo ký tự, đổi ký tự thành số bằng `int('7')`, `str.count`, **mảng đếm 10 phần tử** `cnt[d]`, `collections.Counter`, `zip(s, s[1:])` và `itertools.pairwise`, tập hợp (`set`), và `next(generator, mặc_định)`.
3. Đi từ cách **đếm lại bằng `s.count` cho từng cặp** → **đếm một lần bằng mảng 10 phần tử rồi quét cặp** → **`Counter`** → **tập các chữ số "tốt" rồi quét cặp** → **một dòng `next(...)`**.
4. Hiểu mấu chốt: điều kiện "xuất hiện bao nhiêu lần" là về **toàn bộ chuỗi** (kể cả phần phía **sau** cặp), nên **phải đếm xong cả chuỗi trước** rồi mới quét tìm cặp. Đếm kiểu "vừa duyệt vừa đếm" sẽ sai.
5. Tránh các lỗi kinh điển: quên điều kiện `a != b`, so sánh ký tự với số (`'2' == 2` là `False`), trả `None`/số thay vì chuỗi `""`/`"23"`, đếm khi đang duyệt, và truy cập `s[i + 1]` vượt biên ở ký tự cuối.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm các chữ số. Một **cặp hợp lệ** là hai chữ số **kề nhau** trong `s` sao cho:

- Chữ số thứ nhất **không bằng** chữ số thứ hai.
- **Mỗi** chữ số trong cặp xuất hiện trong `s` **đúng bằng** giá trị số của nó (ví dụ `'2'` phải xuất hiện đúng `2` lần).

Trả về **cặp hợp lệ đầu tiên** tìm thấy khi duyệt `s` **từ trái sang phải**. Nếu không có cặp hợp lệ nào, trả về **chuỗi rỗng**.

**Ví dụ 1**

- Input: `s = "2523533"`
- Output: `"23"`
- Giải thích: chữ số `'2'` xuất hiện `2` lần, chữ số `'3'` xuất hiện `3` lần. Mỗi chữ số trong cặp `"23"` xuất hiện trong `s` đúng bằng giá trị số của nó, nên đáp án là `"23"`.

**Ví dụ 2**

- Input: `s = "221"`
- Output: `"21"`
- Giải thích: chữ số `'2'` xuất hiện `2` lần và chữ số `'1'` xuất hiện `1` lần.

**Ví dụ 3**

- Input: `s = "22"`
- Output: `""`
- Giải thích: không có cặp kề nhau hợp lệ nào (hai chữ số `2` bằng nhau).

**Ràng buộc**

- `2 <= s.length <= 100`
- `s` chỉ gồm các chữ số từ `'1'` đến `'9'`.

In [1]:
examples = [
    ("2523533", "23"),
    ("221", "21"),
    ("22", ""),
]

for s, expected in examples:
    print(repr(s), "-> kỳ vọng", repr(expected))

'2523533' -> kỳ vọng '23'
'221' -> kỳ vọng '21'
'22' -> kỳ vọng ''


## 3. Kiến thức nền

### 3.1. Chuỗi là dãy ký tự: duyệt, chỉ số, `len`

Chuỗi `s = "221"` có `len(s) = 3`; `s[0] = '2'`, `s[2] = '1'`. Có thể duyệt từng ký tự bằng `for ch in s`. **Cặp kề nhau** là `(s[i], s[i + 1])`; vì `s[i + 1]` phải tồn tại nên `i` chỉ đến `len(s) - 2`, tức `range(len(s) - 1)`.

In [2]:
s = "2523533"
print(len(s), s[0], s[-1])                       # 7 2 3
for i in range(len(s) - 1):
    print(f"i={i}: cặp kề = {s[i] + s[i + 1]!r}  (hai ký tự {s[i]!r} và {s[i + 1]!r})")

7 2 3
i=0: cặp kề = '25'  (hai ký tự '2' và '5')
i=1: cặp kề = '52'  (hai ký tự '5' và '2')
i=2: cặp kề = '23'  (hai ký tự '2' và '3')
i=3: cặp kề = '35'  (hai ký tự '3' và '5')
i=4: cặp kề = '53'  (hai ký tự '5' và '3')
i=5: cặp kề = '33'  (hai ký tự '3' và '3')


### 3.2. Ký tự khác số: `'7'` và `7`

Phần tử của chuỗi là **ký tự** (chuỗi dài 1): `'7'` khác với số nguyên `7`. So sánh `'7' == 7` cho `False`! Muốn đổi, dùng `int('7')` ra `7`. Ở bài này, "giá trị số" của chữ số `'3'` là `int('3') = 3` — ta cần đổi trước khi so sánh với số lần xuất hiện.

In [3]:
print('7' == 7, int('7') == 7)                    # False True
print(int('3') + 1, '3' + '1')                    # 4 31   (cộng số khác với nối chuỗi)
print([int(ch) for ch in "2523533"])              # [2, 5, 2, 3, 5, 3, 3]

False True
4 31
[2, 5, 2, 3, 5, 3, 3]


### 3.3. `str.count`: đếm số lần xuất hiện

`s.count('2')` trả về **số lần** ký tự (hoặc chuỗi con) `'2'` xuất hiện trong `s`. Mỗi lần gọi quét cả chuỗi nên tốn `O(n)`.

In [4]:
s = "2523533"
print(s.count('2'), s.count('3'), s.count('5'), s.count('9'))      # 2 3 2 0

2 3 2 0


### 3.4. Mảng đếm 10 phần tử: `cnt[d]`

Chữ số chỉ có `10` giá trị (`0..9`), nên ta đếm bằng một list `cnt = [0] * 10`, trong đó `cnt[d]` = số lần chữ số `d` xuất hiện. Duyệt `s` một lần: `cnt[int(ch)] += 1`. Sau đó mọi câu hỏi "chữ số `d` xuất hiện mấy lần?" trả lời trong `O(1)` bằng `cnt[d]`.

In [5]:
s = "2523533"
cnt = [0] * 10
for ch in s:
    cnt[int(ch)] += 1
print(cnt)                                        # [0, 0, 2, 3, 0, 2, 0, 0, 0, 0]
print("chữ số 2 xuất hiện", cnt[2], "lần; chữ số 3 xuất hiện", cnt[3], "lần")

[0, 0, 2, 3, 0, 2, 0, 0, 0, 0]
chữ số 2 xuất hiện 2 lần; chữ số 3 xuất hiện 3 lần


### 3.5. `collections.Counter`: đếm nhanh bằng thư viện

`Counter("2523533")` tạo ra một từ điển đếm theo **ký tự** (khóa là `'2'`, `'3'`, ...). Truy cập khóa không tồn tại cho `0` (không báo lỗi). Khi dùng, nhớ khóa là **ký tự** chứ không phải số.

In [6]:
from collections import Counter

c = Counter("2523533")
print(c)                                          # Counter({'3': 3, '2': 2, '5': 2})
print(c['3'], c['9'])                             # 3 0   (khóa vắng mặt cho 0)
print(c['3'] == int('3'))                         # True

Counter({'3': 3, '2': 2, '5': 2})
3 0
True


### 3.6. Ghép cặp kề: `zip(s, s[1:])` và `pairwise`

`s[1:]` là chuỗi bỏ ký tự đầu. `zip(s, s[1:])` ghép `s[0]` với `s[1]`, `s[1]` với `s[2]`, ... và dừng khi hết chuỗi ngắn hơn — đúng các **cặp kề**. `itertools.pairwise(s)` làm điều tương tự. Cả hai tránh phải viết chỉ số `i + 1` (và tránh lỗi vượt biên).

In [7]:
from itertools import pairwise

s = "221"
print(list(zip(s, s[1:])))                        # [('2', '2'), ('2', '1')]
print(list(pairwise(s)))                          # [('2', '2'), ('2', '1')]

[('2', '2'), ('2', '1')]
[('2', '2'), ('2', '1')]


### 3.7. Tập hợp `set` và `next(gen, mặc_định)`

- `set` lưu các phần tử **không trùng**, kiểm tra `x in tập` trong `O(1)`. Có thể dựng bằng *set comprehension* `{d for d in ... if điều_kiện}`.
- `next(it, mặc_định)` lấy phần tử **đầu tiên** của iterator `it`; nếu iterator rỗng thì trả `mặc_định` (không báo lỗi). Kết hợp với generator, ta được "phần tử đầu tiên thỏa điều kiện, hoặc giá trị mặc định" trong một dòng.

In [8]:
good = {d for d in range(1, 10) if d % 3 == 0}
print(good, 3 in good, 4 in good)                 # {3, 6, 9} True False

print(next((x for x in [1, 4, 6, 7] if x % 2 == 0), "không có"))     # 4  (phần tử chẵn đầu tiên)
print(next((x for x in [1, 3, 5] if x % 2 == 0), "không có"))        # không có

{9, 3, 6} True False
4
không có


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "2523533"`.

**Bước 1 — đếm mỗi chữ số trong toàn chuỗi:**

| chữ số | `2` | `3` | `5` |
|---|---|---|---|
| số lần xuất hiện | 2 | 3 | 2 |
| bằng giá trị số? | có (2 = 2) | có (3 = 3) | **không** (2 ≠ 5) |

Vậy chữ số "tốt" (số lần xuất hiện = giá trị) là `2` và `3`. Chữ số `5` không tốt.

**Bước 2 — quét các cặp kề từ trái sang phải.** Chuỗi là `2, 5, 2, 3, 5, 3, 3` nên các cặp kề là `(2,5)`, `(5,2)`, `(2,3)`, `(3,5)`, `(5,3)`, `(3,3)`:

| `i` | cặp | hai chữ số khác nhau? | cả hai "tốt"? | hợp lệ? |
|---|---|---|---|---|
| 0 | `(2,5)` | có | `5` không tốt | không |
| 1 | `(5,2)` | có | `5` không tốt | không |
| 2 | `(2,3)` | có | `2` tốt, `3` tốt | **có** |

Cặp **`(2,3)`** ở `i = 2` là cặp đầu tiên thỏa mọi điều kiện ⇒ đáp án `"23"`. (Các cặp phía sau không cần xét nữa.)

**Điều then chốt:** việc quyết định "tốt" cần **đếm toàn chuỗi** (kể cả phần phía sau cặp), nên ta đếm xong **trước** rồi mới quét.

In [9]:
s = "2523533"
cnt = [0] * 10
for ch in s:
    cnt[int(ch)] += 1

for d in range(1, 10):
    if cnt[d] > 0:
        print(f"chữ số {d}: xuất hiện {cnt[d]} lần -> {'tốt' if cnt[d] == d else 'không tốt'}")

print()
for i in range(len(s) - 1):
    a, b = int(s[i]), int(s[i + 1])
    ok = a != b and cnt[a] == a and cnt[b] == b
    print(f"i={i}: cặp ({a},{b}): khác nhau={a != b}, cnt[{a}]={cnt[a]}, cnt[{b}]={cnt[b]} -> {'HỢP LỆ' if ok else 'không'}")

chữ số 2: xuất hiện 2 lần -> tốt
chữ số 3: xuất hiện 3 lần -> tốt
chữ số 5: xuất hiện 2 lần -> không tốt

i=0: cặp (2,5): khác nhau=True, cnt[2]=2, cnt[5]=2 -> không
i=1: cặp (5,2): khác nhau=True, cnt[5]=2, cnt[2]=2 -> không
i=2: cặp (2,3): khác nhau=True, cnt[2]=2, cnt[3]=3 -> HỢP LỆ
i=3: cặp (3,5): khác nhau=True, cnt[3]=3, cnt[5]=2 -> không
i=4: cặp (5,3): khác nhau=True, cnt[5]=2, cnt[3]=3 -> không
i=5: cặp (3,3): khác nhau=False, cnt[3]=3, cnt[3]=3 -> không


## 5. Cách 1 — Brute Force / baseline: dùng `s.count` cho từng cặp

**Ý tưởng.** Duyệt các cặp kề `(s[i], s[i + 1])` từ trái sang phải. Với mỗi cặp, kiểm tra ba điều kiện bằng định nghĩa:

1. `s[i] != s[i + 1]`;
2. `s.count(s[i]) == int(s[i])`;
3. `s.count(s[i + 1]) == int(s[i + 1])`.

Cặp đầu tiên thỏa cả ba thì trả về `s[i] + s[i + 1]`. Hết vòng lặp mà không thấy thì trả `""`.

In [10]:
def find_valid_pair_count(s: str) -> str:
    for i in range(len(s) - 1):
        a, b = s[i], s[i + 1]
        if a != b and s.count(a) == int(a) and s.count(b) == int(b):
            return a + b                      # cặp hợp lệ đầu tiên
    return ""


for s, expected in examples:
    print(repr(find_valid_pair_count(s)), "(kỳ vọng", repr(expected), ")")

'23' (kỳ vọng '23' )
'21' (kỳ vọng '21' )
'' (kỳ vọng '' )


**Độ phức tạp (Cách 1).** Mỗi cặp gọi `count` hai lần, mỗi lần `O(n)`; có `n - 1` cặp ⇒ **Time `O(n²)`**, **Space `O(1)`**. Với `n <= 100` thì chỉ khoảng `200 × 100 = 20 000` bước — rất nhanh; nhưng ta đang **đếm lại cùng một chữ số nhiều lần**.

**Nút thắt:** chỉ có `9` chữ số khác nhau, mà ta gọi `count` cỡ `2n` lần. Đếm **một lần cho tất cả** rồi tra cứu sẽ gọn hơn nhiều.

## 6. Ý tưởng thuật toán / bước đột phá

**Hai giai đoạn tách bạch:**

1. **Đếm** số lần xuất hiện của mọi chữ số trong `s` — một lượt, `O(n)`, lưu vào mảng `cnt` (10 phần tử) hoặc `Counter`.
2. **Quét cặp kề** từ trái sang phải; với mỗi cặp `(x, y)` kiểm tra `x != y`, `cnt[x] == x`, `cnt[y] == y` trong `O(1)`; cặp đầu tiên thỏa thì trả về.

Vì sao phải đếm **trước**? Vì điều kiện "xuất hiện bao nhiêu lần" là về **toàn chuỗi**. Nếu đếm dần khi duyệt thì lúc xét cặp ở vị trí `i`, ta mới biết số lần xuất hiện **tính đến `i`**, thiếu các lần xuất hiện phía sau. Ví dụ `s = "2523533"`: khi đến cặp `(2, 3)` ở `i = 2`, chữ số `3` mới xuất hiện `1` lần (ở vị trí 3) hoặc `0` lần tính trước đó, trong khi tổng cộng nó xuất hiện `3` lần.

In [11]:
# Minh họa: đếm "dần" (sai) vs đếm "trước" (đúng) tại cặp (2,3) ở i = 2 của "2523533"
s = "2523533"
i = 2
dem_dan = [0] * 10
for ch in s[:i + 2]:                              # chỉ đếm đến hết cặp hiện tại
    dem_dan[int(ch)] += 1
dem_truoc = [0] * 10
for ch in s:                                      # đếm cả chuỗi
    dem_truoc[int(ch)] += 1

print("đếm dần  : cnt[2] =", dem_dan[2], ", cnt[3] =", dem_dan[3], "-> thiếu các lần xuất hiện phía sau")
print("đếm trước: cnt[2] =", dem_truoc[2], ", cnt[3] =", dem_truoc[3], "-> đúng yêu cầu của đề")

đếm dần  : cnt[2] = 2 , cnt[3] = 1 -> thiếu các lần xuất hiện phía sau
đếm trước: cnt[2] = 2 , cnt[3] = 3 -> đúng yêu cầu của đề


## 7. Các cách tiếp theo

### Cách 2 — Mảng đếm `cnt` 10 phần tử rồi quét cặp

**Giải thích từng đoạn.**

1. `cnt = [0] * 10` rồi `for ch in s: cnt[int(ch)] += 1` — đếm toàn chuỗi.
2. `for i in range(len(s) - 1)`: lấy `x = int(s[i])`, `y = int(s[i + 1])` (đổi sang số để dùng làm chỉ số và so sánh).
3. Điều kiện `x != y and cnt[x] == x and cnt[y] == y` ⇒ `return s[i:i + 2]` (cắt hai ký tự liên tiếp, cho chuỗi dài 2).
4. Hết vòng lặp ⇒ `return ""`.

In [12]:
def find_valid_pair_array(s: str) -> str:
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1                             # cnt[d] = số lần chữ số d xuất hiện
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return s[i:i + 2]
    return ""


for s, expected in examples:
    print(repr(find_valid_pair_array(s)), "(kỳ vọng", repr(expected), ")")

'23' (kỳ vọng '23' )
'21' (kỳ vọng '21' )
'' (kỳ vọng '' )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionArray:
    def findValidPair(self, s: str) -> str:
        cnt = [0] * 10
        for ch in s:
            cnt[int(ch)] += 1
        for i in range(len(s) - 1):
            x, y = int(s[i]), int(s[i + 1])
            if x != y and cnt[x] == x and cnt[y] == y:
                return s[i:i + 2]
        return ""


print(repr(SolutionArray().findValidPair("2523533")))        # '23'

'23'


**Độ phức tạp Cách 2:** **Time `O(n)`** (đếm `O(n)` + quét `O(n)`), **Space `O(1)`** (mảng cố định 10 phần tử).

### Cách 3 — `Counter` + `zip(s, s[1:])`

**Giải thích.** `Counter(s)` đếm theo **ký tự** nên khi tra cứu phải đổi `int(ch)` để so với số lần. `zip(s, s[1:])` cho từng cặp ký tự kề nhau `(a, b)`; ta không cần chỉ số `i`, cũng không có nguy cơ vượt biên.

In [14]:
def find_valid_pair_counter(s: str) -> str:
    cnt = Counter(s)                                   # khóa là ký tự '1'..'9'
    for a, b in zip(s, s[1:]):
        if a != b and cnt[a] == int(a) and cnt[b] == int(b):
            return a + b
    return ""


for s, expected in examples:
    print(repr(find_valid_pair_counter(s)), "(kỳ vọng", repr(expected), ")")

'23' (kỳ vọng '23' )
'21' (kỳ vọng '21' )
'' (kỳ vọng '' )


In [15]:
# Code nộp LeetCode cho Cách 3
from collections import Counter


class SolutionCounter:
    def findValidPair(self, s: str) -> str:
        cnt = Counter(s)
        for a, b in zip(s, s[1:]):
            if a != b and cnt[a] == int(a) and cnt[b] == int(b):
                return a + b
        return ""


print(repr(SolutionCounter().findValidPair("221")))          # '21'

'21'


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (tối đa 9 khóa; `s[1:]` tạo thêm một chuỗi `O(n)` tạm thời, nếu muốn tránh hãy dùng `pairwise`).

### Cách 4 — Lập sẵn tập "chữ số tốt" rồi quét cặp

**Giải thích.** Chữ số `d` là **tốt** khi `s.count(str(d)) == d`. Tìm tập các chữ số tốt **một lần**: `good = {d for d in range(1, 10) if cnt[d] == d}`. Khi đó cặp `(x, y)` hợp lệ **khi và chỉ khi** `x != y` và `x`, `y` **cùng nằm trong `good`** — điều kiện gọn và dễ giải thích hơn. Nếu `len(good) < 2` thì chắc chắn không có cặp hợp lệ và có thể trả `""` ngay.

In [16]:
def find_valid_pair_good_set(s: str) -> str:
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    good = {d for d in range(1, 10) if cnt[d] == d}    # các chữ số "tốt"
    if len(good) < 2:
        return ""                                       # cần ít nhất hai chữ số tốt khác nhau
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and x in good and y in good:
            return s[i:i + 2]
    return ""


for s, expected in examples:
    print(repr(find_valid_pair_good_set(s)), "(kỳ vọng", repr(expected), ")")

print("tập chữ số tốt của '2523533':", {d for d in range(1, 10) if "2523533".count(str(d)) == d})

'23' (kỳ vọng '23' )
'21' (kỳ vọng '21' )
'' (kỳ vọng '' )
tập chữ số tốt của '2523533': {2, 3}


In [17]:
# Code nộp LeetCode cho Cách 4
class SolutionGoodSet:
    def findValidPair(self, s: str) -> str:
        good = {d for d in range(1, 10) if s.count(str(d)) == d}
        for i in range(len(s) - 1):
            x, y = int(s[i]), int(s[i + 1])
            if x != y and x in good and y in good:
                return s[i:i + 2]
        return ""


print(repr(SolutionGoodSet().findValidPair("2523533")))      # '23'

'23'


**Độ phức tạp Cách 4:** **Time `O(n)`** (ở bản nộp dùng `s.count` cho `9` chữ số: `9 × O(n)` = `O(n)`), **Space `O(1)`** (tập có tối đa `9` phần tử).

### Cách 5 — Một dòng: `pairwise` + `next`

**Giải thích.** Gom hai giai đoạn vào một biểu thức: `cnt = Counter(map(int, s))` đếm theo **số** (khóa là `int`), `pairwise(map(int, s))` cho các cặp số kề nhau, generator lọc các cặp thỏa điều kiện, `next(..., None)` lấy cặp đầu tiên; cuối cùng chuyển cặp thành chuỗi hoặc trả `""`. Rất gọn, nhưng khó đọc hơn — nên hiểu các cách trước rồi mới dùng.

In [18]:
def find_valid_pair_oneliner(s: str) -> str:
    cnt = Counter(map(int, s))                          # khóa là số nguyên
    pair = next(((x, y) for x, y in pairwise(map(int, s)) if x != y and cnt[x] == x and cnt[y] == y), None)
    return "" if pair is None else f"{pair[0]}{pair[1]}"


for s, expected in examples:
    print(repr(find_valid_pair_oneliner(s)), "(kỳ vọng", repr(expected), ")")

'23' (kỳ vọng '23' )
'21' (kỳ vọng '21' )
'' (kỳ vọng '' )


In [19]:
# Code nộp LeetCode cho Cách 5
from collections import Counter
from itertools import pairwise


class SolutionOneLiner:
    def findValidPair(self, s: str) -> str:
        cnt = Counter(map(int, s))
        for x, y in pairwise(map(int, s)):
            if x != y and cnt[x] == x and cnt[y] == y:
                return f"{x}{y}"
        return ""


print(repr(SolutionOneLiner().findValidPair("22")))          # ''

''


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

Bản nộp mình chọn là **Cách 2** (mảng đếm 10 phần tử): ngắn, `O(n)`, bộ nhớ cố định và dạy đúng kỹ thuật "**đếm trước, quét sau**" dùng được ở rất nhiều bài khác.

## 8. TRACE từng bước

Trace **Cách 2** (mảng đếm) với `s = "2523533"`. Mảng đếm sau giai đoạn 1: `cnt = [0, 0, 2, 3, 0, 2, 0, 0, 0, 0]` (`cnt[2] = 2`, `cnt[3] = 3`, `cnt[5] = 2`).

| `i` | `x = s[i]` | `y = s[i+1]` | `x != y` | `cnt[x] == x` | `cnt[y] == y` | kết quả |
|---|---|---|---|---|---|---|
| 0 | 2 | 5 | có | 2 == 2 có | 2 == 5 **không** | bỏ |
| 1 | 5 | 2 | có | 2 == 5 **không** | 2 == 2 có | bỏ |
| 2 | 2 | 3 | có | 2 == 2 có | 3 == 3 có | **trả `"23"`** |

Hàm dừng ngay ở `i = 2`, không cần xét các cặp còn lại.

In [20]:
def trace_array(s: str) -> str:
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    print(f"s = {s!r}")
    print("cnt =", cnt)
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        c1, c2, c3 = x != y, cnt[x] == x, cnt[y] == y
        print(f"  i={i}: ({x},{y}): x!=y {c1} | cnt[{x}]={cnt[x]}=={x}? {c2} | cnt[{y}]={cnt[y]}=={y}? {c3}")
        if c1 and c2 and c3:
            print(f"  -> cặp hợp lệ đầu tiên: {s[i:i + 2]!r}")
            return s[i:i + 2]
    print("  -> không có cặp hợp lệ")
    return ""


assert trace_array("2523533") == "23"
print()
assert trace_array("22") == ""

s = '2523533'
cnt = [0, 0, 2, 3, 0, 2, 0, 0, 0, 0]
  i=0: (2,5): x!=y True | cnt[2]=2==2? True | cnt[5]=2==5? False
  i=1: (5,2): x!=y True | cnt[5]=2==5? False | cnt[2]=2==2? True
  i=2: (2,3): x!=y True | cnt[2]=2==2? True | cnt[3]=3==3? True
  -> cặp hợp lệ đầu tiên: '23'

s = '22'
cnt = [0, 0, 2, 0, 0, 0, 0, 0, 0, 0]
  i=0: (2,2): x!=y False | cnt[2]=2==2? True | cnt[2]=2==2? True
  -> không có cặp hợp lệ


Trace **Cách 4** (tập chữ số tốt) trên `s = "221"` và `s = "122"` để thấy cách `good` giúp loại bỏ nhanh các cặp.

In [21]:
def trace_good_set(s: str) -> str:
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    good = {d for d in range(1, 10) if cnt[d] == d}
    print(f"s = {s!r}, cnt = {[(d, cnt[d]) for d in range(1, 10) if cnt[d]]}, good = {sorted(good)}")
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        hop_le = x != y and x in good and y in good
        print(f"  i={i}: ({x},{y}) -> {'HỢP LỆ' if hop_le else 'không'}")
        if hop_le:
            return s[i:i + 2]
    return ""


assert trace_good_set("221") == "21"
print()
assert trace_good_set("122") == "12"

s = '221', cnt = [(1, 1), (2, 2)], good = [1, 2]
  i=0: (2,2) -> không
  i=1: (2,1) -> HỢP LỆ

s = '122', cnt = [(1, 1), (2, 2)], good = [1, 2]
  i=0: (1,2) -> HỢP LỆ


## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (mảng đếm 10 phần tử): đếm toàn chuỗi trước, quét cặp kề sau.

In [22]:
class Solution:
    def findValidPair(self, s: str) -> str:
        cnt = [0] * 10                                    # cnt[d] = số lần chữ số d xuất hiện trong s
        for ch in s:
            cnt[int(ch)] += 1
        for i in range(len(s) - 1):                       # i + 1 phải còn trong chuỗi
            x, y = int(s[i]), int(s[i + 1])
            if x != y and cnt[x] == x and cnt[y] == y:
                return s[i:i + 2]                         # cặp hợp lệ đầu tiên từ trái sang phải
        return ""


sol = Solution()
for s, expected in examples:
    print(repr(sol.findValidPair(s)), "(kỳ vọng", repr(expected), ")")

'23' (kỳ vọng '23' )
'21' (kỳ vọng '21' )
'' (kỳ vọng '' )


## 10. Vì sao thuật toán đúng

- **Đếm trên toàn chuỗi:** giai đoạn 1 duyệt **mọi** ký tự của `s`, nên `cnt[d]` đúng bằng số lần `d` xuất hiện trong **cả** chuỗi — đúng yêu cầu "xuất hiện trong `s`" của đề (không phải "tính đến vị trí hiện tại").
- **Kiểm tra đủ ba điều kiện:** cặp `(x, y)` hợp lệ khi `x != y` (điều kiện 1) và `cnt[x] == x`, `cnt[y] == y` (điều kiện 2 cho từng chữ số). Mã kiểm tra đúng cả ba.
- **Đúng thứ tự "đầu tiên":** vòng `for i in range(len(s) - 1)` đi từ trái sang phải và `return` ở cặp thỏa đầu tiên, nên không có cặp nào ở bên trái bị bỏ sót hay cặp ở bên phải được chọn nhầm.
- **Không vượt biên:** `i` chạy đến `len(s) - 2` nên `s[i + 1]` luôn tồn tại.
- **Kết quả đúng kiểu:** `s[i:i + 2]` là một **chuỗi** dài 2 (ví dụ `"23"`); không có cặp ⇒ `""`.
- **Chữ số `0`:** đề chỉ có `'1'..'9'`. (Nếu bạn tự thử chuỗi có `'0'`: ký tự `'0'` đã xuất hiện thì `cnt[0] >= 1` khác `0`, nên `'0'` không bao giờ là chữ số "tốt".)

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. `s.count` cho từng cặp | `O(n²)` | `O(1)` |
| 2. Mảng đếm 10 phần tử + quét cặp | `O(n)` | `O(1)` |
| 3. `Counter` + `zip(s, s[1:])` | `O(n)` | `O(1)` (+ `O(n)` tạm cho `s[1:]`) |
| 4. Tập chữ số tốt + quét cặp | `O(n)` | `O(1)` |
| 5. Một dòng `pairwise` + `next` | `O(n)` | `O(1)` |

Vì mọi cách đều phải nhìn mọi ký tự ít nhất một lần (để đếm), `O(n)` là tối ưu.

## 12. Bộ test

Gom các cách vào `impls` rồi so với kỳ vọng. "Trọng tài" `oracle` được viết theo **đúng từng chữ của đề**: liệt kê mọi cặp kề, kiểm tra hai điều kiện bằng `s.count`, lấy cặp đầu tiên.

In [23]:
import random

sol = Solution()
impls = {
    "count": find_valid_pair_count,
    "array": find_valid_pair_array,
    "counter": find_valid_pair_counter,
    "good_set": find_valid_pair_good_set,
    "oneliner": find_valid_pair_oneliner,
    "leetcode": sol.findValidPair,
}


def oracle(s):
    """Trọng tài: liệt kê mọi cặp kề, kiểm tra từng điều kiện theo đề, lấy cặp đầu tiên."""
    valid = []
    for i in range(len(s) - 1):
        a, b = s[i], s[i + 1]
        khac_nhau = (a != b)
        a_du = (s.count(a) == int(a))
        b_du = (s.count(b) == int(b))
        if khac_nhau and a_du and b_du:
            valid.append(a + b)
    return valid[0] if valid else ""


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    assert oracle(s) == expected, (label, "oracle")
    print("PASS", label, repr(s), "->", repr(expected))


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1 '2523533' -> '23'
PASS ví dụ 2 '221' -> '21'
PASS ví dụ 3 '22' -> ''


### 12.1. Các trường hợp biên

Các ví dụ "tự dựng" dưới đây có kỳ vọng tính tay ngay trong comment.

In [24]:
check_all("11", "", "hai chữ số giống nhau (điều kiện a != b)")
check_all("12", "", "'2' chỉ xuất hiện 1 lần, không đủ 2")
check_all("21", "", "'2' chỉ xuất hiện 1 lần")
check_all("122", "12", "'1' xuất hiện 1, '2' xuất hiện 2 -> '12'")
check_all("221", "21", "ví dụ 2 xáo trộn")
check_all("212", "21", "cặp (2,1) ở đầu: 2 xuất hiện 2 lần, 1 lần 1 -> '21'")
check_all("3133", "31", "'3' xuất hiện 3 lần, '1' một lần; cặp (3,1) ở i = 0")
check_all("1332", "", "'1' tốt nhưng '3' xuất hiện 2 lần, '2' 1 lần -> không chữ số nào cặp được")
check_all("3321", "", "tự dựng: '3' xuất hiện 2 lần, '2' một lần -> chỉ '1' tốt (cần hai chữ số tốt khác nhau)")
check_all("33231", "31", "tự dựng: '3' xuất hiện 3 lần, '1' một lần (tốt), '2' một lần (không tốt) -> cặp (3,1) ở cuối")
check_all("9999999991", "91", "'9' xuất hiện 9 lần, '1' một lần: cặp (9,1) ở cuối")
check_all("1999999999", "19", "tự dựng: '1' một lần, '9' chín lần: cặp (1,9) ở đầu")
check_all("5" * 5 + "4" * 4, "54", "tự dựng: '5' xuất hiện 5 lần, '4' xuất hiện 4 lần -> cặp (5,4) ở giữa")

PASS hai chữ số giống nhau (điều kiện a != b) '11' -> ''
PASS '2' chỉ xuất hiện 1 lần, không đủ 2 '12' -> ''
PASS '2' chỉ xuất hiện 1 lần '21' -> ''
PASS '1' xuất hiện 1, '2' xuất hiện 2 -> '12' '122' -> '12'
PASS ví dụ 2 xáo trộn '221' -> '21'
PASS cặp (2,1) ở đầu: 2 xuất hiện 2 lần, 1 lần 1 -> '21' '212' -> '21'
PASS '3' xuất hiện 3 lần, '1' một lần; cặp (3,1) ở i = 0 '3133' -> '31'
PASS '1' tốt nhưng '3' xuất hiện 2 lần, '2' 1 lần -> không chữ số nào cặp được '1332' -> ''
PASS tự dựng: '3' xuất hiện 2 lần, '2' một lần -> chỉ '1' tốt (cần hai chữ số tốt khác nhau) '3321' -> ''
PASS tự dựng: '3' xuất hiện 3 lần, '1' một lần (tốt), '2' một lần (không tốt) -> cặp (3,1) ở cuối '33231' -> '31'
PASS '9' xuất hiện 9 lần, '1' một lần: cặp (9,1) ở cuối '9999999991' -> '91'
PASS tự dựng: '1' một lần, '9' chín lần: cặp (1,9) ở đầu '1999999999' -> '19'
PASS tự dựng: '5' xuất hiện 5 lần, '4' xuất hiện 4 lần -> cặp (5,4) ở giữa '555554444' -> '54'


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách sai hay gặp: (a) **đếm dần khi duyệt** (không đếm trước); (b) **bỏ điều kiện `a != b`**; (c) **trả về cặp cuối cùng** thay vì đầu tiên (hoặc kiểm tra các cặp từ phải sang trái).

In [25]:
def wrong_count_while_scanning(s):
    cnt = [0] * 10
    for i in range(len(s) - 1):
        cnt[int(s[i])] += 1                              # SAI: chỉ đếm những gì đã đi qua
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return s[i:i + 2]
    return ""


def wrong_no_distinct(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if cnt[x] == x and cnt[y] == y:                  # SAI: thiếu x != y
            return s[i:i + 2]
    return ""


def wrong_last_pair(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    ans = ""
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            ans = s[i:i + 2]                             # SAI: ghi đè -> lấy cặp CUỐI
    return ans


print(repr(wrong_count_while_scanning("2523533")), "!=", repr(sol.findValidPair("2523533")))
print(repr(wrong_no_distinct("22")), "!=", repr(sol.findValidPair("22")))
assert wrong_count_while_scanning("2523533") != sol.findValidPair("2523533")
assert wrong_no_distinct("22") != sol.findValidPair("22")

'' != '23'
'22' != ''


Để minh họa `wrong_last_pair`, ta cần một chuỗi có **ít nhất hai cặp hợp lệ khác nhau** (khi đó "cặp đầu tiên" khác "cặp cuối cùng"). Cell sau tìm một chuỗi như vậy bằng vét cạn trên các chuỗi nhỏ gồm chữ số `1, 2, 3`.

In [26]:
import itertools

# Tìm chuỗi nhỏ có >= 2 cặp hợp lệ để chứng minh "lấy cặp cuối" khác "lấy cặp đầu"
found = None
for n in range(2, 9):
    for tup in itertools.product("123", repeat=n):
        s = "".join(tup)
        valid = [s[i:i + 2] for i in range(len(s) - 1)
                 if s[i] != s[i + 1] and s.count(s[i]) == int(s[i]) and s.count(s[i + 1]) == int(s[i + 1])]
        if len(set(valid)) >= 2:
            found = (s, valid)
            break
    if found:
        break
print("chuỗi:", found[0], "| các cặp hợp lệ theo thứ tự:", found[1])
s = found[0]
print("đúng (đầu tiên):", repr(sol.findValidPair(s)), "| sai (cuối cùng):", repr(wrong_last_pair(s)))
assert sol.findValidPair(s) != wrong_last_pair(s)

chuỗi: 212 | các cặp hợp lệ theo thứ tự: ['21', '12']
đúng (đầu tiên): '21' | sai (cuối cùng): '12'


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

(1) **Ngẫu nhiên thuần:** `s` độ dài `2..100` với chữ số `1..9` (đúng ràng buộc). Chuỗi ngẫu nhiên thuần hiếm khi có cặp hợp lệ nên ta (2) **sinh có chủ đích**: chọn vài chữ số `d`, đặt `d` bản sao mỗi chữ số, thêm vài "nhiễu", rồi xáo trộn (đảm bảo độ dài `<= 100`). (3) **Vét cạn:** mọi chuỗi độ dài `2..7` với chữ số trong `{1, 2, 3}`.

In [27]:
rng = random.Random(3438)
non_empty = 0

# (1) ngẫu nhiên thuần
for trial in range(1500):
    n = rng.randint(2, 100)
    s = "".join(rng.choice("123456789") for _ in range(n))
    ref = oracle(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
    non_empty += bool(ref)

# (2) sinh có chủ đích để có nhiều cặp hợp lệ
for trial in range(2500):
    digits = rng.sample(range(1, 10), rng.randint(1, 5))
    chars = []
    for d in digits:
        chars += [str(d)] * d                          # đúng d lần
    for _ in range(rng.randint(0, 6)):
        chars.append(str(rng.randint(1, 9)))           # nhiễu có thể làm một số chữ số không còn "tốt"
    rng.shuffle(chars)
    if not 2 <= len(chars) <= 100:
        continue
    s = "".join(chars)
    ref = oracle(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
    non_empty += bool(ref)
print("PASS ngẫu nhiên; số chuỗi có đáp án khác rỗng:", non_empty)
assert non_empty > 200, "bộ sinh cần tạo đủ nhiều ca có đáp án"

# (3) vét cạn
count_strings = 0
for n in range(2, 8):
    for tup in itertools.product("123", repeat=n):
        s = "".join(tup)
        ref = oracle(s)
        for key, fn in impls.items():
            assert fn(s) == ref, (key, s)
        count_strings += 1
print("PASS vét cạn", count_strings, "chuỗi nhỏ")

PASS ngẫu nhiên; số chuỗi có đáp án khác rỗng: 1832


PASS vét cạn 3276 chuỗi nhỏ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm "dần" khi duyệt thay vì đếm trước

In [28]:
def sai_dem_dan(s):
    cnt = [0] * 10
    for i in range(len(s) - 1):
        cnt[int(s[i])] += 1                               # SAI: chưa thấy các lần xuất hiện phía sau
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return s[i:i + 2]
    return ""


print("sai:", repr(sai_dem_dan("2523533")), "| đúng:", repr(Solution().findValidPair("2523533")))
print("sai:", repr(sai_dem_dan("221")), "| đúng:", repr(Solution().findValidPair("221")))

sai: '' | đúng: '23'
sai: '' | đúng: '21'


**Vì sao sai:** đề yêu cầu số lần xuất hiện trong **cả chuỗi `s`**. Khi đang đứng ở cặp `i`, bạn mới thấy phần chuỗi đã đi qua; các lần xuất hiện phía sau (ví dụ hai chữ `3` cuối của `"2523533"`) chưa được đếm. **Cách sửa:** vòng lặp thứ nhất đếm hết `s`, vòng lặp thứ hai mới quét cặp.

### Lỗi 2: Quên điều kiện hai chữ số khác nhau

In [29]:
def sai_quen_khac_nhau(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if cnt[x] == x and cnt[y] == y:                   # SAI: thiếu x != y
            return s[i:i + 2]
    return ""


print("sai:", repr(sai_quen_khac_nhau("22")), "| đúng:", repr(Solution().findValidPair("22")))
assert sai_quen_khac_nhau("22") == "22" and Solution().findValidPair("22") == ""

sai: '22' | đúng: ''


**Vì sao sai:** với `"22"`, chữ số `2` xuất hiện đúng `2` lần nên `cnt[2] == 2` đúng; cặp `(2, 2)` lọt qua và hàm trả `"22"` — **sai**, đây chính là ví dụ 3 của đề (đáp án đúng là `""` vì hai chữ số trong cặp phải khác nhau). **Cách sửa:** thêm `x != y`.

### Lỗi 3: So sánh ký tự với số (`'2' == 2`)

In [30]:
def sai_so_ky_tu_voi_so(s):
    cnt = Counter(s)                                      # khóa là KÝ TỰ
    for a, b in zip(s, s[1:]):
        if a != b and cnt[a] == a and cnt[b] == b:        # SAI: cnt[a] là số, a là ký tự '2' -> luôn False
            return a + b
    return ""


print("sai:", repr(sai_so_ky_tu_voi_so("2523533")), "| đúng:", repr(Solution().findValidPair("2523533")))
print("'2' == 2 ->", '2' == 2, "| int('2') == 2 ->", int('2') == 2)

sai: '' | đúng: '23'
'2' == 2 -> False | int('2') == 2 -> True


**Vì sao sai:** phần tử của chuỗi là **ký tự** (`'2'`), còn `cnt[a]` là **số nguyên**; `2 == '2'` luôn `False` nên không bao giờ tìm thấy cặp. Python không báo lỗi — chỉ âm thầm trả `""`. **Cách sửa:** `cnt[a] == int(a)`, hoặc đổi cả chuỗi sang số từ đầu (`map(int, s)`).

### Lỗi 4: Vượt biên khi truy cập `s[i + 1]`

In [31]:
def sai_vuot_bien(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    for i in range(len(s)):                               # SAI: i = len(s) - 1 thì s[i + 1] không tồn tại
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return s[i:i + 2]
    return ""


try:
    sai_vuot_bien("22")
except IndexError as e:
    print("IndexError:", e)

IndexError: string index out of range


**Vì sao sai:** cặp kề `(s[i], s[i + 1])` cần `i + 1 <= len(s) - 1`, tức `i <= len(s) - 2`. **Cách sửa:** `range(len(s) - 1)`, hoặc dùng `zip(s, s[1:])` / `pairwise(s)` để Python tự lo biên.

### Lỗi 5: Trả về `None` hoặc số thay vì chuỗi

In [32]:
def sai_tra_none(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return int(s[i:i + 2])                        # SAI: trả số 23 thay vì chuỗi "23"
    # SAI: không có return "" ở cuối -> trả None


print(repr(sai_tra_none("2523533")), "| đúng:", repr(Solution().findValidPair("2523533")))
print(repr(sai_tra_none("22")), "| đúng:", repr(Solution().findValidPair("22")))

23 | đúng: '23'
None | đúng: ''


**Vì sao sai:** đề yêu cầu trả về **chuỗi** (`"23"`) và **chuỗi rỗng** `""` khi không có đáp án. Trả `23` hoặc `None` sẽ bị chấm sai kiểu. **Cách sửa:** luôn `return s[i:i + 2]` và `return ""` ở cuối.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện kiểu "**xuất hiện bao nhiêu lần trong cả chuỗi**" ⇒ **đếm trước** (mảng đếm / `Counter`), **rồi mới** xử lý từng vị trí.
- Chữ số chỉ có `10` giá trị ⇒ dùng mảng đếm `[0] * 10` với `int(ch)` làm chỉ số (hoặc `[0] * 26` với `ord(ch) - ord('a')` cho chữ cái thường).
- "**Cặp kề nhau**" ⇒ `zip(s, s[1:])`, `pairwise(s)` hoặc `range(len(s) - 1)`; tránh viết `s[i + 1]` với `i` chạy đến cuối.
- "**Cặp/phần tử đầu tiên thỏa điều kiện, nếu không thì mặc định**" ⇒ vòng lặp với `return` sớm, hoặc `next(generator, mặc_định)`.
- Nhớ `'2'` (ký tự) khác `2` (số): đổi bằng `int(ch)` trước khi so sánh hoặc làm số học.
- Bài đếm tần suất thường có pattern chung: **một lượt đếm + một lượt kiểm tra**. Tự đặt câu hỏi: "thông tin toàn cục nào cần biết trước khi quyết định?"

## 15. Biến thể của bài

1. **Trả về cặp hợp lệ cuối cùng** (duyệt từ phải sang trái, hoặc lưu lại khi gặp).
2. **Đếm số cặp hợp lệ** thay vì trả cặp đầu tiên.
3. **Trả về vị trí `i`** của cặp hợp lệ đầu tiên (hoặc `-1`).
4. **Danh sách mọi chữ số "tốt"** (số lần xuất hiện bằng giá trị) của chuỗi.
5. **Chuỗi gồm chữ cái:** "chữ cái nào xuất hiện đúng bằng thứ tự của nó trong bảng chữ cái". Các bài liên quan: First Unique Character in a String (LeetCode 387), Find the Difference (LeetCode 389), Majority Element (LeetCode 169).

In [33]:
def last_valid_pair(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    ans = ""
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            ans = s[i:i + 2]
    return ans


def count_valid_pairs(s):
    cnt = Counter(map(int, s))
    return sum(1 for x, y in pairwise(map(int, s)) if x != y and cnt[x] == x and cnt[y] == y)


def index_of_first_valid_pair(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return i
    return -1


def good_digits(s):
    return sorted({int(ch) for ch in s if s.count(ch) == int(ch)})


s = "2523533"
print(last_valid_pair(s), count_valid_pairs(s), index_of_first_valid_pair(s), good_digits(s))
assert last_valid_pair("2332") == ""                        # '2' xuất hiện 2, '3' xuất hiện 2: chỉ 2 tốt
assert count_valid_pairs("2523533") == 1                    # chỉ cặp (2,3) ở i = 2
assert index_of_first_valid_pair("2523533") == 2
assert index_of_first_valid_pair("22") == -1
assert good_digits("2523533") == [2, 3]

23 1 2 [2, 3]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `s.count` mỗi cặp | đếm lại cho từng cặp | `O(n²)` | `O(1)` | bám sát đề, làm trọng tài, `n` nhỏ |
| 2. Mảng đếm 10 phần tử | đếm một lần, tra `cnt[d]` | `O(n)` | `O(1)` | **nên nộp**: nhanh, bộ nhớ cố định |
| 3. `Counter` + `zip` | đếm theo ký tự, ghép cặp | `O(n)` | `O(1)` | viết nhanh, quen thư viện |
| 4. Tập chữ số tốt | tìm sẵn `good`, cặp = hai phần tử `good` khác nhau | `O(n)` | `O(1)` | dễ giải thích; loại sớm nếu `len(good) < 2` |
| 5. Một dòng `pairwise` + `next` | gộp đếm và tìm cặp đầu | `O(n)` | `O(1)` | khi đã quen generator |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"3133".count("3")`, `int("8") + 1`, `list(zip("abc", "abc"[1:]))`, `Solution().findValidPair("122")`, `Solution().findValidPair("11")`.

**Bài 2 (tính tay).** Với `s = "3133"`: lập bảng đếm từng chữ số, xác định chữ số nào "tốt", rồi liệt kê các cặp kề và tìm cặp hợp lệ đầu tiên.

**Bài 3 (điền TODO).** Hoàn thành `find_valid_pair_todo(s)` theo Cách 2 (mảng đếm 10 phần tử).

In [34]:
def find_valid_pair_todo(s):
    cnt = [0] * 10
    # TODO 1: duyệt từng ch trong s, tăng cnt[int(ch)] lên 1
    # TODO 2: for i in range(len(s) - 1): x, y = int(s[i]), int(s[i + 1])
    # TODO 3: nếu x != y và cnt[x] == x và cnt[y] == y thì return s[i:i + 2]
    # TODO 4: nếu hết vòng lặp, return ""
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `find_valid_pair_from_scratch(s)` theo **Cách 3** (`Counter` + `zip`) **mà không dùng** `Counter`: tự tạo từ điển `cnt = {}` và đếm bằng `cnt[ch] = cnt.get(ch, 0) + 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `find_all_valid_pairs(s)` trả về **danh sách mọi cặp hợp lệ** theo thứ tự xuất hiện (cho phép trùng nhau). (b) Viết `valid_pair_letters(s)` cho chuỗi **chữ cái thường**: chữ cái thứ `k` trong bảng chữ cái (`'a'` là `1`, `'b'` là `2`, ...) "tốt" khi nó xuất hiện đúng `k` lần; trả về cặp kề đầu tiên gồm hai chữ cái **khác nhau**, cùng "tốt" (hoặc `""`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`, `9`, `[('a', 'b'), ('b', 'c')]`, `"12"`, `""`. Bài 3: `cnt[int(ch)] += 1` trong vòng lặp đầu tiên.
- **Mức 2 (vừa):** Bài 2: `cnt[3] = 3`, `cnt[1] = 1` ⇒ cả hai "tốt". Các cặp: `(3,1)` (hợp lệ ngay ở `i = 0`) ⇒ `"31"`. Bài 4: `cnt.get(ch, 0)` trả `0` nếu khóa chưa có.
- **Mức 3 (sâu):** Bài 5(a): thay `return` bằng `ans.append(...)`, trả `ans` sau vòng lặp. 5(b): `k = ord(ch) - ord('a') + 1`, điều kiện `cnt[ch] == k`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
assert "3133".count("3") == 3
assert int("8") + 1 == 9
assert list(zip("abc", "abc"[1:])) == [('a', 'b'), ('b', 'c')]
assert Solution().findValidPair("122") == "12"
assert Solution().findValidPair("11") == ""

# Bài 2
assert Solution().findValidPair("3133") == "31"


# Bài 3
def find_valid_pair_todo_ans(s):
    cnt = [0] * 10
    for ch in s:
        cnt[int(ch)] += 1
    for i in range(len(s) - 1):
        x, y = int(s[i]), int(s[i + 1])
        if x != y and cnt[x] == x and cnt[y] == y:
            return s[i:i + 2]
    return ""


for s, expected in examples:
    assert find_valid_pair_todo_ans(s) == expected


# Bài 4
def find_valid_pair_from_scratch(s):
    cnt = {}
    for ch in s:
        cnt[ch] = cnt.get(ch, 0) + 1
    for a, b in zip(s, s[1:]):
        if a != b and cnt[a] == int(a) and cnt[b] == int(b):
            return a + b
    return ""


for s, expected in examples:
    assert find_valid_pair_from_scratch(s) == expected


# Bài 5
def find_all_valid_pairs(s):
    cnt = Counter(s)
    return [a + b for a, b in zip(s, s[1:]) if a != b and cnt[a] == int(a) and cnt[b] == int(b)]


def valid_pair_letters(s):
    cnt = Counter(s)
    for a, b in zip(s, s[1:]):
        ka, kb = ord(a) - ord("a") + 1, ord(b) - ord("a") + 1
        if a != b and cnt[a] == ka and cnt[b] == kb:
            return a + b
    return ""


assert find_all_valid_pairs("2523533") == ["23"]
assert find_all_valid_pairs("2132") == ["21"]
assert valid_pair_letters("abb") == "ab"                      # 'a' xuất hiện 1 lần (k=1), 'b' xuất hiện 2 lần (k=2)
assert valid_pair_letters("aa") == ""
assert valid_pair_letters("bba") == "ba"
print("Tất cả đáp án đều đúng")

Tất cả đáp án đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm toàn chuỗi trước, rồi quét cặp kề)

```python
class Solution:
    def findValidPair(self, s: str) -> str:
        cnt = [0] * 10
        for ch in s:
            cnt[int(ch)] += 1                       # bước 1: đếm TOÀN chuỗi
        for i in range(len(s) - 1):                 # bước 2: quét cặp kề, từ trái sang phải
            x, y = int(s[i]), int(s[i + 1])
            if x != y and cnt[x] == x and cnt[y] == y:
                return s[i:i + 2]
        return ""
```

**Checklist**

- [ ] Đã **đếm xong toàn chuỗi trước** khi quét cặp (không đếm dần)?
- [ ] Có điều kiện hai chữ số **khác nhau** (`x != y`)?
- [ ] So sánh **số với số** (`cnt[x] == x` với `x = int(...)`), không so ký tự với số?
- [ ] Vòng lặp cặp dừng ở `len(s) - 2` (không vượt biên) hoặc dùng `zip`/`pairwise`?
- [ ] Trả về **cặp đầu tiên** từ trái sang phải, dạng **chuỗi** dài 2; không có thì `""`?
- [ ] Đã thử `"22"` (cặp bằng nhau), một chuỗi không có đáp án và chuỗi có đáp án ở cuối?